# Attention Mechanisms from Scratch

**Module 06: Attention Mechanism**  
**Objective**: Understand attention and its variants from first principles

**Problem with RNNs**: Fixed-size context vector bottlenecks information

**Solution**: Attention allows model to "focus" on relevant parts

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

## What You'll Learn
1. Attention intuition and mathematics
2. Bahdanau (additive) attention
3. Luong (multiplicative) attention  
4. Scaled dot-product attention
5. Multi-head attention
6. Self-attention

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

## 1. Attention Intuition

**Traditional Seq2Seq**: Encoder → Fixed vector → Decoder

**Problem**: Long sequences → information loss

**Attention**: Decoder attends to all encoder states dynamically

**Key idea**: Weighted combination of encoder outputs
$$c_t = \sum_{i=1}^{T_x} \alpha_{ti} h_i$$

where $\alpha_{ti}$ are attention weights (what to focus on).

In [ ]:
def visualize_attention_concept():
    """Visualize attention weights."""
    
    # Source sentence: "The cat sat on the mat"
    # Target sentence: "Le chat était assis sur le tapis"
    
    source = ["The", "cat", "sat", "on", "the", "mat"]
    target = ["Le", "chat", "était", "assis", "sur", "le", "tapis"]
    
    # Simulated attention weights (target attends to source)
    attention = np.array([
        [0.8, 0.1, 0.0, 0.0, 0.1, 0.0],  # "Le" → "The"
        [0.1, 0.8, 0.0, 0.0, 0.1, 0.0],  # "chat" → "cat"
        [0.0, 0.1, 0.7, 0.1, 0.0, 0.1],  # "était" → "sat"
        [0.0, 0.1, 0.8, 0.0, 0.0, 0.1],  # "assis" → "sat"
        [0.0, 0.0, 0.1, 0.8, 0.0, 0.1],  # "sur" → "on"
        [0.7, 0.0, 0.0, 0.1, 0.2, 0.0],  # "le" → "the"
        [0.1, 0.0, 0.0, 0.0, 0.1, 0.8],  # "tapis" → "mat"
    ])
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(attention, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=source, yticklabels=target,
                cbar_kws={'label': 'Attention Weight'})
    plt.xlabel('Source (English)')
    plt.ylabel('Target (French)')
    plt.title('Attention Visualization: Translation Task')
    plt.tight_layout()
    plt.show()
    
    print("✓ Each target word attends to relevant source words!")

visualize_attention_concept()

## 2. Bahdanau Attention (Additive)

**Alignment score** (how well input $h_j$ matches output $s_i$):
$$e_{ij} = v^T \tanh(W_1 s_{i-1} + W_2 h_j)$$

**Attention weights** (normalize):
$$\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k=1}^{T_x} \exp(e_{ik})}$$

**Context vector**:
$$c_i = \sum_{j=1}^{T_x} \alpha_{ij} h_j$$

In [ ]:
class BahdanauAttention:
    """Bahdanau (additive) attention."""
    
    def __init__(self, hidden_dim, attention_dim):
        self.hidden_dim = hidden_dim
        self.attention_dim = attention_dim
        
        # Learnable parameters
        self.W1 = np.random.randn(attention_dim, hidden_dim) * 0.01  # For decoder state
        self.W2 = np.random.randn(attention_dim, hidden_dim) * 0.01  # For encoder states
        self.v = np.random.randn(attention_dim, 1) * 0.01  # Attention vector
    
    def compute_attention(self, decoder_state, encoder_states):
        """
        Compute attention weights.
        
        Args:
            decoder_state: (hidden_dim, 1) - current decoder state
            encoder_states: (T_x, hidden_dim) - all encoder states
        
        Returns:
            attention_weights: (T_x,) - normalized weights
            context: (hidden_dim, 1) - weighted sum of encoder states
        """
        T_x = encoder_states.shape[0]
        
        # Compute alignment scores
        scores = []
        for j in range(T_x):
            h_j = encoder_states[j:j+1].T  # (hidden_dim, 1)
            
            # e_ij = v^T * tanh(W1 * s + W2 * h_j)
            score = self.v.T @ np.tanh(self.W1 @ decoder_state + self.W2 @ h_j)
            scores.append(score[0, 0])
        
        scores = np.array(scores)
        
        # Softmax
        exp_scores = np.exp(scores - np.max(scores))
        attention_weights = exp_scores / np.sum(exp_scores)
        
        # Context vector
        context = np.zeros((self.hidden_dim, 1))
        for j in range(T_x):
            context += attention_weights[j] * encoder_states[j:j+1].T
        
        return attention_weights, context

# Test Bahdanau attention
print("Testing Bahdanau Attention\n")

hidden_dim = 64
attention_dim = 32
seq_len = 10

bahdanau = BahdanauAttention(hidden_dim, attention_dim)

# Random encoder states and decoder state
encoder_states = np.random.randn(seq_len, hidden_dim)
decoder_state = np.random.randn(hidden_dim, 1)

# Compute attention
attention_weights, context = bahdanau.compute_attention(decoder_state, encoder_states)

print(f"Encoder sequence length: {seq_len}")
print(f"Attention weights shape: {attention_weights.shape}")
print(f"Context vector shape: {context.shape}")
print(f"\nAttention weights: {attention_weights}")
print(f"Sum of weights: {np.sum(attention_weights):.4f} (should be 1.0)")

# Visualize
plt.figure(figsize=(12, 4))
plt.bar(range(seq_len), attention_weights)
plt.xlabel('Encoder Position')
plt.ylabel('Attention Weight')
plt.title('Bahdanau Attention Weights')
plt.grid(axis='y', alpha=0.3)
plt.show()

## 3. Luong Attention (Multiplicative)

**Simpler alternative to Bahdanau**

**Alignment score**:
- **Dot**: $e_{ij} = s_i^T h_j$
- **General**: $e_{ij} = s_i^T W h_j$
- **Concat**: Same as Bahdanau

**Key difference**: Uses current decoder state (not previous)

In [ ]:
class LuongAttention:
    """Luong (multiplicative) attention."""
    
    def __init__(self, hidden_dim, method='general'):
        self.hidden_dim = hidden_dim
        self.method = method
        
        if method == 'general':
            self.W = np.random.randn(hidden_dim, hidden_dim) * 0.01
    
    def compute_attention(self, decoder_state, encoder_states):
        """
        Compute attention weights.
        
        Args:
            decoder_state: (hidden_dim, 1)
            encoder_states: (T_x, hidden_dim)
        
        Returns:
            attention_weights, context
        """
        T_x = encoder_states.shape[0]
        
        # Compute alignment scores
        if self.method == 'dot':
            # e_ij = s_i^T * h_j
            scores = encoder_states @ decoder_state  # (T_x, 1)
            scores = scores.squeeze()
        
        elif self.method == 'general':
            # e_ij = s_i^T * W * h_j
            scores = encoder_states @ self.W @ decoder_state  # (T_x, 1)
            scores = scores.squeeze()
        
        # Softmax
        exp_scores = np.exp(scores - np.max(scores))
        attention_weights = exp_scores / np.sum(exp_scores)
        
        # Context vector
        context = np.zeros((self.hidden_dim, 1))
        for j in range(T_x):
            context += attention_weights[j] * encoder_states[j:j+1].T
        
        return attention_weights, context

# Test Luong attention
print("\nTesting Luong Attention (General)\n")

luong = LuongAttention(hidden_dim, method='general')

attention_weights_luong, context_luong = luong.compute_attention(decoder_state, encoder_states)

print(f"Attention weights shape: {attention_weights_luong.shape}")
print(f"Context vector shape: {context_luong.shape}")
print(f"Sum of weights: {np.sum(attention_weights_luong):.4f}")

# Compare Bahdanau vs Luong
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(seq_len), attention_weights)
axes[0].set_title('Bahdanau Attention')
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Weight')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(range(seq_len), attention_weights_luong)
axes[1].set_title('Luong Attention')
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Weight')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Scaled Dot-Product Attention

**Foundation of Transformers**

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**Components**:
- **Query** ($Q$): What am I looking for?
- **Key** ($K$): What do I contain?
- **Value** ($V$): What do I output?

**Scaling factor** $\sqrt{d_k}$: Prevents softmax saturation for large $d_k$

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled dot-product attention.
    
    Args:
        Q: (batch, seq_len_q, d_k) queries
        K: (batch, seq_len_k, d_k) keys
        V: (batch, seq_len_v, d_v) values
        mask: optional mask
    
    Returns:
        output: (batch, seq_len_q, d_v)
        attention_weights: (batch, seq_len_q, seq_len_k)
    """
    d_k = Q.shape[-1]
    
    # Compute attention scores: Q @ K^T / sqrt(d_k)
    scores = Q @ K.transpose(0, 2, 1) / np.sqrt(d_k)  # (batch, seq_q, seq_k)
    
    # Apply mask if provided
    if mask is not None:
        scores = scores + mask * -1e9
    
    # Softmax
    exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    attention_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
    
    # Apply attention to values
    output = attention_weights @ V  # (batch, seq_q, d_v)
    
    return output, attention_weights

# Test scaled dot-product attention
print("\nTesting Scaled Dot-Product Attention\n")

batch_size = 2
seq_len_q = 5
seq_len_k = 8
d_k = 64
d_v = 64

Q = np.random.randn(batch_size, seq_len_q, d_k)
K = np.random.randn(batch_size, seq_len_k, d_k)
V = np.random.randn(batch_size, seq_len_k, d_v)

output, attention_weights = scaled_dot_product_attention(Q, K, V)

print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attention_weights.shape}")

# Visualize attention for first batch
plt.figure(figsize=(10, 6))
sns.heatmap(attention_weights[0], annot=True, fmt='.2f', cmap='Blues',
            xticklabels=[f'K{i}' for i in range(seq_len_k)],
            yticklabels=[f'Q{i}' for i in range(seq_len_q)])
plt.xlabel('Keys (Source)')
plt.ylabel('Queries (Target)')
plt.title('Scaled Dot-Product Attention Weights')
plt.tight_layout()
plt.show()

print(f"\n✓ Each query attends to all keys with learned weights!")

## 5. Self-Attention

**Key innovation**: Input serves as Q, K, and V

$$\text{Self-Attention}(X) = \text{Attention}(XW_Q, XW_K, XW_V)$$

**Purpose**: Each position attends to all positions in same sequence

**Applications**:
- BERT: Bidirectional context
- GPT: Causal (masked) self-attention

In [ ]:
class SelfAttention:
    """Self-attention layer."""
    
    def __init__(self, d_model, d_k):
        self.d_model = d_model
        self.d_k = d_k
        
        # Linear projections
        self.W_q = np.random.randn(d_model, d_k) * 0.01
        self.W_k = np.random.randn(d_model, d_k) * 0.01
        self.W_v = np.random.randn(d_model, d_k) * 0.01
    
    def forward(self, X, mask=None):
        """
        Self-attention forward pass.
        
        Args:
            X: (batch, seq_len, d_model) input
            mask: optional mask for causal attention
        
        Returns:
            output: (batch, seq_len, d_k)
            attention_weights: (batch, seq_len, seq_len)
        """
        # Project to Q, K, V
        Q = X @ self.W_q  # (batch, seq_len, d_k)
        K = X @ self.W_k
        V = X @ self.W_v
        
        # Scaled dot-product attention
        output, attention_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        return output, attention_weights

# Test self-attention
print("\nTesting Self-Attention\n")

batch_size = 1
seq_len = 6
d_model = 128
d_k = 64

self_attn = SelfAttention(d_model, d_k)

X = np.random.randn(batch_size, seq_len, d_model)
output, attention_weights = self_attn.forward(X)

print(f"Input shape: {X.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attention_weights.shape}")

# Visualize self-attention
plt.figure(figsize=(8, 6))
sns.heatmap(attention_weights[0], annot=True, fmt='.2f', cmap='YlGnBu',
            xticklabels=[f'Pos{i}' for i in range(seq_len)],
            yticklabels=[f'Pos{i}' for i in range(seq_len)])
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.title('Self-Attention: Each Position Attends to All Positions')
plt.tight_layout()
plt.show()

print(f"\n✓ Self-attention: Input attends to itself!")

## 6. Multi-Head Attention

**Idea**: Multiple attention heads learn different aspects

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$$

where 
$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

**Benefits**:
- Different heads focus on different relationships
- Increases model capacity
- Standard in Transformers (8-16 heads)

In [ ]:
class MultiHeadAttention:
    """Multi-head attention."""
    
    def __init__(self, d_model, num_heads):
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Projection matrices for all heads
        self.W_q = np.random.randn(d_model, d_model) * 0.01
        self.W_k = np.random.randn(d_model, d_model) * 0.01
        self.W_v = np.random.randn(d_model, d_model) * 0.01
        self.W_o = np.random.randn(d_model, d_model) * 0.01
    
    def split_heads(self, x, batch_size):
        """Split last dimension into (num_heads, d_k)."""
        x = x.reshape(batch_size, -1, self.num_heads, self.d_k)
        return x.transpose(0, 2, 1, 3)  # (batch, heads, seq_len, d_k)
    
    def forward(self, Q, K, V, mask=None):
        """
        Multi-head attention forward.
        
        Args:
            Q, K, V: (batch, seq_len, d_model)
        
        Returns:
            output: (batch, seq_len, d_model)
            attention_weights: list of (batch, seq_len, seq_len) for each head
        """
        batch_size = Q.shape[0]
        
        # Linear projections
        Q = Q @ self.W_q  # (batch, seq_len, d_model)
        K = K @ self.W_k
        V = V @ self.W_v
        
        # Split into heads
        Q = self.split_heads(Q, batch_size)  # (batch, heads, seq_len, d_k)
        K = self.split_heads(K, batch_size)
        V = self.split_heads(V, batch_size)
        
        # Apply attention for each head
        d_k = self.d_k
        scores = np.matmul(Q, K.transpose(0, 1, 3, 2)) / np.sqrt(d_k)
        
        if mask is not None:
            scores = scores + mask * -1e9
        
        exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
        attention_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
        
        output = np.matmul(attention_weights, V)  # (batch, heads, seq_len, d_k)
        
        # Concatenate heads
        output = output.transpose(0, 2, 1, 3)  # (batch, seq_len, heads, d_k)
        output = output.reshape(batch_size, -1, self.d_model)  # (batch, seq_len, d_model)
        
        # Final linear
        output = output @ self.W_o
        
        return output, attention_weights

# Test multi-head attention
print("\nTesting Multi-Head Attention\n")

d_model = 128
num_heads = 8
batch_size = 2
seq_len = 10

mha = MultiHeadAttention(d_model, num_heads)

Q = np.random.randn(batch_size, seq_len, d_model)
K = np.random.randn(batch_size, seq_len, d_model)
V = np.random.randn(batch_size, seq_len, d_model)

output, attention_weights = mha.forward(Q, K, V)

print(f"d_model: {d_model}")
print(f"num_heads: {num_heads}")
print(f"d_k per head: {mha.d_k}")
print(f"\nInput Q shape: {Q.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attention_weights.shape}  # (batch, heads, seq, seq)")

# Visualize different heads
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for head in range(num_heads):
    ax = axes[head]
    sns.heatmap(attention_weights[0, head], cmap='viridis', cbar=False, ax=ax)
    ax.set_title(f'Head {head+1}')
    ax.set_xlabel('Key')
    ax.set_ylabel('Query')

plt.suptitle('Multi-Head Attention: Different Heads Learn Different Patterns', fontsize=16)
plt.tight_layout()
plt.show()

print(f"\n✓ Each head can focus on different relationships!")

## Summary

You've mastered attention mechanisms from scratch:
- ✅ Bahdanau (additive) attention
- ✅ Luong (multiplicative) attention
- ✅ Scaled dot-product attention
- ✅ Self-attention
- ✅ Multi-head attention

**Key insights**:
1. Attention = weighted combination of values based on query-key similarity
2. Self-attention allows each position to attend to all positions
3. Scaling prevents softmax saturation ($\sqrt{d_k}$)
4. Multi-head attention learns diverse relationships
5. Attention is the foundation of Transformers

**Attention types**:
- **Cross-attention**: Attend to different sequence (encoder-decoder)
- **Self-attention**: Attend to same sequence
- **Causal attention**: Mask future positions (GPT)
- **Bidirectional**: See all positions (BERT)

**Next Notebook**: Implementing Transformers with PyTorch

## Exercises

1. **Causal Mask**: Implement causal self-attention for autoregressive models
2. **Relative Positional Encoding**: Add relative position bias to attention
3. **Local Attention**: Restrict attention to nearby positions (efficient for long sequences)